# 🎤 K-pop 월드투어 공연 실적 분석 과제

**출제:** BI팀 시니어 분석가 → 신입 분석가  
**용도:** 주간 경영진 비즈니스 리뷰(WBR) 투어 파트

## 상황 설명

우리 회사 소속 아티스트 5팀(NOVA, LUMINA, BLAZE, ECHO9, STARLIT)이 2024~2025년 2년 동안 진행한 월드투어 공연 250회의 실적 데이터입니다. 경영진은 다음 달 **2026~2027 투어 계획 회의**에서 세 가지를 결정해야 합니다.

1. 어느 권역에 공연을 더 배정할 것인가
2. 티켓 가격을 어디서 올리고 어디서 유지할 것인가
3. 아티스트별로 어느 규모의 공연장을 잡을 것인가

여러분의 대시보드가 이 회의의 근거 자료가 됩니다. 그리고 **이 데이터는 현업 데이터처럼 지저분합니다.** 정제하지 않고 그리면 경영진에게 틀린 숫자가 올라갑니다.

## 0. 환경 세팅

> ⚠️ **시작 전 확인:** `business_data.csv`는 이 노트북이 아니라 `make_business_data.py`를 **터미널에서 한 번 실행**해서 만듭니다.  
> (`python make_business_data.py`) 두 파일과 이 노트북은 **같은 폴더**에 있어야 합니다.

In [3]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 주의: sns.set_theme()는 폰트 설정을 초기화하므로 반드시 '먼저' 호출할 것
sns.set_theme(style="whitegrid")

if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":          # Mac
    plt.rcParams["font.family"] = "AppleGothic"

plt.rcParams["axes.unicode_minus"] = False   # 음수 기호(-) 깨짐 방지

### 데이터 사전

| 컬럼 | 타입 | 설명 |
|---|---|---|
| `show_date` | 날짜 | 공연일 (2024-01 ~ 2025-12) |
| `artist` | 범주 | 아티스트 (STARLIT은 신인급) |
| `region` | 범주 | 권역: 국내, 일본, 동남아, 북미, 유럽, 중남미 |
| `venue_type` | 범주 | 공연장 규모: 홀, 아레나, 스타디움 |
| `capacity` | 정수 | 공연장 수용 인원 (명) |
| `avg_ticket_price` | 실수 | 평균 티켓 가격 (**USD 기준으로 집계되어야 함**) |
| `occupancy_rate` | 실수 | 좌석 점유율 (%) = 판매 좌석 ÷ 수용 인원 × 100 |
| `revenue` | 실수 | 공연 티켓 매출 (USD) |

> 💡 **힌트가 되는 관계식:** 정상적인 행이라면  
> `revenue ≈ capacity × (occupancy_rate / 100) × avg_ticket_price`  
> 가 대략 성립해야 합니다. 이 식은 과제 전체에서 쓸모가 있습니다.

In [5]:
df = pd.read_csv("business_data.csv", parse_dates=["show_date"])
df.head()

,show_date,artist,region,venue_type,capacity,avg_ticket_price,occupancy_rate,revenue
0,2024-01-04,LUMINA,동남아,아레나,10100,107.8,78.4,844951.0
1,2024-01-07,BLAZE,동남아,홀,4600,87.4,84.3,344846.0
2,2024-01-08,ECHO9,북미,홀,2200,134.4,74.3,225553.0
3,2024-01-15,NOVA,국내,아레나,11800,136.4,76.9,1235072.0
4,2024-01-20,NOVA,국내,스타디움,56600,123.1,76.8,5199264.0


---
# 과제 1. 투어 매출은 어느 권역에 쏠려 있는가?

> **경영진 질문:** "내년에 공연을 어디에 더 넣어야 하지? 지금 돈은 어디서 벌고 있어?"

### 비즈니스 문제 의도
권역별 매출을 막대 차트로 보여주는 것 자체는 쉽습니다. 이 과제의 진짜 목적은 **"차트를 그리기 전에 데이터를 믿어도 되는지 확인하는 습관"** 입니다. 단 한 건의 잘못된 행이 권역 순위를 뒤집어 놓으면, 경영진은 엉뚱한 곳에 공연을 배정하게 됩니다. 또한 매출 총액만 보면 "공연을 많이 해서 매출이 큰 권역"과 "공연 1회당 효율이 좋은 권역"이 구분되지 않습니다.

### 분석 포인트 힌트
- 결측치는 **컬럼별 개수와 비율(%)** 두 가지로 확인하세요. 비율이 몇 % 수준인지가 처리 방식을 결정합니다.
- `describe()`에서 **min / max** 를 보고 "이 값이 물리적으로, 비즈니스적으로 가능한가?"를 컬럼마다 스스로 물어보세요. 점유율의 최대값은 몇이어야 할까요? 일본 공연 티켓이 USD로 얼마쯤이면 자연스러울까요?
- `revenue`의 이상치는 min/max만으로는 판단하기 어렵습니다. 위의 관계식으로 **기대 매출**을 직접 계산해서 실제 매출과 비교해 보세요.
- 이상치는 무조건 삭제하는 것이 아닙니다. **입력 오류로 보이는 것 / 수정 가능한 것 / 제외해야 하는 것**을 구분하고, 판단 근거를 한 줄씩 남기세요.
- 차트는 **총매출**과 **공연 1회당 평균 매출** 두 개를 나란히 그려야 쏠림의 원인을 설명할 수 있습니다. 공연 횟수(countplot)도 함께 보면 좋습니다.

In [8]:
# ── 1-1. 기본 구조 확인 ──────────────────────
# TODO: 행/열 수, 컬럼 타입 확인
print(df.shape)
print(df.dtypes)

(250, 8)
show_date           datetime64[us]
artist                         str
region                         str
venue_type                     str
capacity                     int64
avg_ticket_price           float64
occupancy_rate             float64
revenue                    float64
dtype: object


In [ ]:
# ── 1-2. 결측치 확인 ─────────────────────────
missing = pd.DataFrame({
    "결측수": df.isna().sum(),                      # TODO: 컬럼별 결측 개수
    "결측비율(%)":(df.isna().sum() / len(df) *100).round(1) ,      # TODO: 컬럼별 결측 비율(%) - 소수 1자리
})
print(missing[missing["결측수"] > 0])

                  결측수  결측비율(%)
venue_type          8      3.2
avg_ticket_price   10      4.0
occupancy_rate     12      4.8


In [22]:
# ── 1-3-a. 수치형 컬럼 기술통계 ──────────────
# TODO: 수치형 컬럼 기술통계 확인
df.describe()

,show_date,capacity,avg_ticket_price,occupancy_rate,revenue
count,250,250.000000,240.000000,238.000000,2.500000e+02
mean,2025-01-11 09:47:31.200000,16907.200000,191.899583,75.265126,1.851518e+06
min,2024-01-04 00:00:00,2000.000000,62.000000,54.200000,1.365860e+05
25%,2024-07-12 12:00:00,4900.000000,104.000000,69.425000,5.200200e+05
50%,2024-12-15 12:00:00,13350.000000,122.850000,75.450000,1.177526e+06
75%,2025-07-28 06:00:00,18100.000000,143.700000,80.675000,1.886510e+06
max,2025-12-26 00:00:00,59900.000000,15800.000000,128.400000,4.425062e+07
std,NaN,14714.605896,1012.224998,8.708859,3.190634e+06


In [38]:
# ── 1-3-b. 이상치 탐색 ───────────────────────
# TODO: 비즈니스적으로 불가능한 점유율 행 찾기
invalid_occ = df[df["occupancy_rate"] > 100]
print(invalid_occ)

# TODO: 티켓 가격이 비정상적으로 높은 행 찾기 (기준값은 직접 정하고 근거를 주석으로)
suspicious_price = df[df["avg_ticket_price"]>1000]    #대부분의 티켓가격은 수백달러임 1000달러는 너무 비정상적으로 높음
print(suspicious_price)

    show_date  artist region venue_type  capacity  avg_ticket_price  \
40 2024-06-01  LUMINA     일본       스타디움     33100             141.8   

    occupancy_rate    revenue  
40           128.4  3202505.0  
    show_date artist region venue_type  capacity  avg_ticket_price  \
29 2024-05-08  ECHO9     일본        아레나      9300           15800.0   

    occupancy_rate   revenue  
29            68.7  860867.0  


In [39]:
# ── 1-3-c. 기대 매출 대비 실제 매출 ──────────
# TODO: 기대 매출 계산 → 배율이 극단적인 행 찾기 (큰 쪽, 작은 쪽 둘 다 확인해볼 것)
df["expected_revenue"] = (df["occupancy_rate"] / 100) * df["capacity"] * df["avg_ticket_price"]
df["revenue_ratio"] = df["revenue"] / df["expected_revenue"]
cols = ["region", "capacity", "occupancy_rate", "avg_ticket_price",
        "revenue", "expected_revenue", "revenue_ratio"]
print(df.sort_values("revenue_ratio", ascending=False)[cols].head())   # 큰 쪽
print(df.sort_values("revenue_ratio", ascending=True)[cols].head())    # 작은 쪽

    region  capacity  occupancy_rate  avg_ticket_price     revenue  \
150    동남아     58400            75.0             102.4  44250620.0   
156     일본      3800            65.2             122.0    311314.0   
102     북미     14500            83.1             146.1   1812522.0   
151    동남아     15700            90.7              95.4   1398312.0   
169     일본     17500            74.4             137.0   1835958.0   

     expected_revenue  revenue_ratio  
150        4485120.00       9.866095  
156         302267.20       1.029930  
102        1760431.95       1.029589  
151        1358486.46       1.029316  
169        1783740.00       1.029274  
    region  capacity  occupancy_rate  avg_ticket_price    revenue  \
29      일본      9300            68.7           15800.0   860867.0   
40      일본     33100           128.4             141.8  3202505.0   
192    중남미     34000            68.8             114.0  2585234.0   
138    동남아     50900            60.4             110.0  3283966.0   


### 📝 이상치 판단 메모 (1-4 정제 전에 먼저 작성)

| 행 번호 | 컬럼 | 값 | 의심 이유 | 처리 방법 (수정 / NaN / 제외) |
|---|---|---|---|---|
| 150 | revenue | 44250620.0 | 값이 비정상적으로 높음 | 역산한값으로 수정|
| 29 | avg_ticket_price | 15800.0 | 값이 비정상적으로 높음 | /10으로 수정|
| 40 | occupancy_rate | 128.4 | 값이 비정상적으로 높음 | 환율로나눠 USD로 환산|

**결측치 처리 방침:** (여기에 작성)


In [ ]:
# ── 1-4. 정제 ───────────────────────────────
df_clean = df.copy()
# TODO: 위 메모대로 이상치를 처리하는 코드 작성
#   (행마다 판단 근거를 주석으로 남길 것)
df_clean["revenue"]=df_clean["revenue"].fillna(df_clean.groupby(["occupancy_rate", "avg_ticket_price"])["revenue"].transform("median"))

In [ ]:
# ── 1-5. 권역별 핵심 지표 집계 ───────────────
region_summary = (
    df_clean.groupby(___)
    .agg(
        total_revenue=(___, ___),
        show_count=(___, ___),
        avg_revenue_per_show=(___, ___),
    )
    .sort_values(___, ascending=False)
)
region_summary

# TIP: 같은 집계를 df(정제 전)로도 한 번 돌려서 순위를 비교해 보세요 → 제출 질문 1번

In [ ]:
# ── 1-6. 시각화 ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# TODO: (좌) 권역별 총매출 (백만 USD 단위로 변환해서 표시)
sns.barplot(___, ax=axes[0])
axes[0].set_title("권역별 총 투어 매출 (백만 USD)")

# TODO: (중) 권역별 공연 횟수 - countplot 사용, 좌측 차트와 같은 권역 순서 유지
sns.countplot(___, ax=axes[1])

# TODO: (우) 권역별 공연 1회당 평균 매출
sns.barplot(___, ax=axes[2])

plt.tight_layout()
plt.show()

### ✍️ 과제 1 제출 답변

**1. 정제 전과 후에 권역별 매출 순위가 바뀌었나요? 바뀌었다면 어떤 행 때문인가요?**  
→ 

**2. 매출 1위 권역은 "공연을 많이 해서"인가요, "1회당 효율이 좋아서"인가요?**  
→ 

**3. 결측치는 어떻게 처리했고, 왜 그렇게 했나요?**  
→ 

**📊 경영진용 인사이트 3줄**  
1.  
2.  
3.  

---
# 과제 2. 티켓 가격을 올리면 좌석이 안 팔리는가?

> **경영진 질문:** "북미 티켓 너무 싼 거 아니야? 가격 올려도 괜찮을까?"

### 비즈니스 문제 의도
가격 정책 의사결정의 핵심은 **가격 탄력성**, 즉 가격이 오를 때 수요(점유율)가 얼마나 빠지는가입니다. 산점도로 두 지표의 관계를 보되, 공연장 규모라는 제3의 변수를 색으로 구분해야 "가격이 비싸서 안 팔린 것"인지 "공연장이 너무 커서 안 팔린 것"인지를 분리할 수 있습니다. 동시에, 산점도는 과제 1에서 놓친 이상치가 가장 잘 드러나는 차트이기도 합니다.

### 분석 포인트 힌트
- 먼저 **정제 전 데이터로 한 번** 그려보세요. 점 하나 때문에 나머지 점들이 한 구석에 뭉개지는 현상이 보인다면, 그게 바로 이상치가 시각화를 망치는 방식입니다.
- 권역마다 물가와 기준 가격이 다릅니다. 국내 150달러와 북미 150달러는 같은 의미가 아닙니다. **권역 평균 대비 상대 가격(가격 프리미엄)** 을 만들면 관계가 더 선명해질 수 있습니다.
- `hue`에 `venue_type`을 넣고, 필요하면 `style`이나 `size`로 차원을 하나 더 추가해 보세요.
- `sns.regplot` 또는 `np.polyfit`으로 추세선을 얹고, `corr()`로 상관계수를 **숫자로도** 제시하세요. 경영진은 "관계가 있다"보다 "상관계수 -0.3"을 더 신뢰합니다.
- 점유율이 높은데 가격도 권역 평균보다 높은 공연이 있다면, 그 아티스트·권역은 가격 인상 여력이 있는 곳입니다.

In [ ]:
# ── 2-1. 정제 전 데이터로 먼저 그려보기 ──────
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=df, x=___, y=___, hue=___, ax=ax)
ax.set_title("[정제 전] 티켓 가격 vs 좌석 점유율")
plt.show()
# TODO: 이 차트에서 무엇이 문제인지 주석으로 적기
# → 

In [ ]:
# ── 2-2. 상대 가격(가격 프리미엄) 만들기 ─────
# TODO: 권역별 평균 티켓가 대비 각 공연의 가격 비율
#   힌트: groupby + transform
df_clean["price_premium"] = ___
df_clean[["region", "avg_ticket_price", "price_premium"]].head()

In [ ]:
# ── 2-3. 정제 후 산점도 ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# TODO: (좌) 절대 가격 vs 점유율, hue=공연장 규모
sns.scatterplot(data=df_clean, x=___, y=___, hue=___, alpha=0.7, ax=axes[0])

# TODO: (우) 가격 프리미엄 vs 점유율, hue=공연장 규모 + 추세선
sns.scatterplot(data=df_clean, x=___, y=___, hue=___, alpha=0.7, ax=axes[1])
sns.regplot(data=df_clean, x=___, y=___, scatter=False, color="black", ax=axes[1])

# TODO: 두 차트의 제목, 축 이름(단위 포함) 설정
___

plt.tight_layout()
plt.show()

In [ ]:
# ── 2-4. 상관계수 ────────────────────────────
# TODO: 전체 / 공연장 규모별 상관계수 비교
print(df_clean[[___, ___]].corr())
print(df_clean.groupby("venue_type")[[___, ___]].corr())

### ✍️ 과제 2 제출 답변

**1. 절대 가격과 상대 가격 중 어느 쪽이 점유율과 더 뚜렷한 관계를 보이나요? 이유는?**  
→ 

**2. 공연장 규모에 따라 가격-점유율 관계가 달라지나요?**  
→ 

**3. "가격을 올려도 될 것 같은" 권역이나 아티스트를 하나 꼽고, 근거를 제시하세요.**  
→ 

**📊 경영진용 인사이트 3줄**  
1.  
2.  
3.  

---
# 과제 3. 어느 아티스트의 흥행이 불안정한가?

> **경영진 질문:** "STARLIT 다음 투어, 아레나로 올려도 될까? 아니면 아직 홀이 맞나?"

### 비즈니스 문제 의도
평균 점유율이 비슷한 두 아티스트라도, 한 팀은 매번 75% 안팎이고 다른 팀은 55%~95%를 오간다면 리스크가 완전히 다릅니다. 공연장 규모를 키우는 결정은 **평균이 아니라 하방(최악의 경우)** 을 봐야 합니다. 박스플롯은 중앙값, 사분위 범위(IQR), 이상치를 한 번에 보여주기 때문에 "안정성" 비교에 적합합니다.

### 분석 포인트 힌트
- 박스의 **높이(IQR)** 가 흥행의 변동성입니다. 박스 아래 수염의 끝이 "최악의 경우"에 가깝습니다.
- `order` 파라미터로 **중앙값 순서**로 정렬하면 비교가 훨씬 쉬워집니다. 알파벳순은 경영진에게 아무 의미가 없습니다.
- 아티스트만으로 나누면 공연장 규모 효과가 섞입니다. `hue="venue_type"`으로 한 번 더 쪼개서 "같은 아레나에서 비교하면 어떤가"를 보세요.
- 그룹별 표본 수가 너무 적은 칸(예: 신인 아티스트의 스타디움 공연)은 결론을 내리기 위험합니다. **표본 수를 반드시 같이 확인**하세요.
- 박스플롯과 바이올린플롯 중 무엇을 쓸지도 여러분의 판단입니다. 선택 이유를 한 줄 적어주세요.

In [ ]:
# ── 3-1. 아티스트별 점유율 요약 통계 ─────────
# TODO: 아티스트별 count, 중앙값, 표준편차, IQR(Q3-Q1) 계산
artist_stats = df_clean.groupby("artist")["occupancy_rate"].agg(
    count=___,
    median=___,
    std=___,
    iqr=lambda x: ___,
)
print(artist_stats.sort_values(___))

# TODO: 중앙값 기준 정렬 순서 리스트 만들기
order = ___

In [ ]:
# ── 3-2. 아티스트별 점유율 분포 ──────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# TODO: (좌) 아티스트별 점유율 박스플롯 (중앙값 순 정렬)
sns.boxplot(data=df_clean, x=___, y=___, order=___, ax=axes[0])
axes[0].set_title("아티스트별 좌석 점유율 분포")

# TODO: (우) 아티스트 × 공연장 규모별 점유율 박스플롯
sns.boxplot(data=df_clean, x=___, y=___, hue=___, order=___,
            hue_order=["홀", "아레나", "스타디움"], ax=axes[1])
axes[1].set_title("아티스트 × 공연장 규모별 좌석 점유율")

# TODO: 점유율 목표선을 수평선으로 추가 - 기준값은 직접 정하고 이유를 주석으로
for ax in axes:
    ax.axhline(___, color="red", linestyle="--", linewidth=1)

plt.tight_layout()
plt.show()

In [ ]:
# ── 3-3. 표본 수 확인 ────────────────────────
# TODO: 아티스트 × 공연장 규모 교차표로 공연 횟수 확인
pd.crosstab(___, ___)

### ✍️ 과제 3 제출 답변

**0. 박스플롯 / 바이올린플롯 중 선택한 것과 이유**  
→ 

**1. 가장 흥행이 안정적인 아티스트와 가장 불안정한 아티스트는 누구인가요?**  
→ 

**2. STARLIT을 아레나로 올리는 것에 대해 찬성 / 반대 / 판단 보류 중 하나를 고르고, 데이터로 근거를 대세요. (표본 수도 고려할 것)**  
→ 

**3. 박스플롯에 표시된 이상치 점 중 과제 1에서 처리하지 않은 것이 있나요? 그것은 오류인가요, 실제 대박/쪽박 공연인가요?**  
→ 

**📊 경영진용 인사이트 3줄**  
1.  
2.  
3.  

---
# 과제 4. 언제, 어디서 매출이 나오는가?

> **경영진 질문:** "2026년 투어 일정을 짜야 하는데, 권역별로 어느 시기에 공연하는 게 좋아?"

### 비즈니스 문제 의도
투어 일정은 **권역 × 시기** 두 축을 동시에 결정해야 합니다. 막대 차트 여러 개로는 이 조합을 한눈에 보기 어렵습니다. 피벗 히트맵은 행(권역)과 열(분기)의 교차 칸에 지표를 색으로 표현해서, "북미는 3분기에 강하다" 같은 패턴을 경영진이 3초 안에 읽을 수 있게 해줍니다. 다만 히트맵은 **어떤 지표를 칸에 넣느냐**에 따라 결론이 완전히 달라집니다.

### 분석 포인트 힌트
- `show_date`에서 **분기**(또는 월)를 파생 컬럼으로 만드세요. `dt.to_period("Q")`가 유용합니다.
- 칸에 넣을 지표를 신중하게 고르세요. **총매출(sum)** 은 "공연을 많이 한 칸"이 진하게 나오고, **1회당 평균 매출(mean)** 은 "효율이 좋은 칸"이 진하게 나옵니다. 둘 중 경영진 질문에 더 맞는 것은 무엇일까요? 둘 다 그려서 비교해보는 것을 추천합니다.
- 피벗 결과에 **빈 칸(NaN)** 이 생길 수 있습니다. "그 분기에 공연이 없었다"는 뜻이므로 0으로 채울지, 비워둘지 판단하세요. 0과 "데이터 없음"은 다른 의미입니다.
- `annot=True`로 숫자를 표시하되, 단위를 백만 USD로 바꾸고 `fmt`로 소수 자릿수를 정리해야 읽힙니다.
- 과제 1에서 이상치를 제대로 처리하지 않았다면, 이 히트맵의 특정 칸 하나가 혼자 튀어 보일 겁니다. 정제 전후를 비교해 보세요.
- **(선택 도전)** 월별 매출 추이를 아티스트별 라인 차트(`sns.lineplot`)로도 그려보세요. 시즌성이 보이나요?

In [ ]:
# ── 4-1. 기간 파생 컬럼 ──────────────────────
# TODO: 분기 컬럼 생성 (예: "2024Q3")
df_clean["quarter"] = ___
df_clean["quarter"].value_counts().sort_index()

In [ ]:
# ── 4-2. 피벗 테이블 ─────────────────────────
# TODO: (A) 권역 × 분기 총매출 (백만 USD)
pivot_total = df_clean.pivot_table(
    index=___, columns=___, values=___, aggfunc=___
) / 1e6

# TODO: (B) 권역 × 분기 공연 1회당 평균 매출 (백만 USD)
pivot_avg = df_clean.pivot_table(
    index=___, columns=___, values=___, aggfunc=___
) / 1e6

# TODO: 권역 순서를 총매출 합계 기준으로 정렬
region_order = ___
pivot_total = pivot_total.loc[region_order]
pivot_avg = pivot_avg.loc[region_order]
pivot_total

In [ ]:
# ── 4-3. 히트맵 ─────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# TODO: (위) 총매출 히트맵 - annot, fmt, cmap, 컬러바 라벨 지정
sns.heatmap(pivot_total, annot=___, fmt=___, cmap=___,
            cbar_kws={"label": ___}, ax=axes[0])
axes[0].set_title("권역 × 분기 총 투어 매출 (백만 USD)")

# TODO: (아래) 1회당 평균 매출 히트맵
sns.heatmap(pivot_avg, ___, ax=axes[1])
axes[1].set_title(___)

plt.tight_layout()
plt.show()

In [ ]:
# ── 4-4. (선택) 월별 추세선 ──────────────────
# TODO: 월 단위 컬럼을 만들고, 아티스트별 월 매출 합계를 lineplot으로
monthly = (
    df_clean.assign(month=___)
    .groupby([___, ___], as_index=False)["revenue"].sum()
)
fig, ax = plt.subplots(figsize=(14, 5))
sns.lineplot(data=monthly, x=___, y=___, hue=___, marker="o", ax=ax)
plt.show()

### ✍️ 과제 4 제출 답변

**1. 총매출 히트맵과 1회당 평균 매출 히트맵에서 가장 진한 칸이 같은가요, 다른가요? 다르다면 왜인가요?**  
→ 

**2. 빈 칸(NaN)을 어떻게 처리했고, 그 선택이 해석에 어떤 영향을 주나요?**  
→ 

**3. 2026년 투어 일정에 대해 경영진에게 한 문장으로 제안한다면?**  
→ 

**📊 경영진용 인사이트 3줄**  
1.  
2.  
3.  